# LLM Fine-Tuning for FPL Prediction

## Notebook 03 -- LoRA Fine-Tuning on Apple Silicon

This notebook documents the process of fine-tuning **Llama 3.2 3B Instruct (4-bit)** using LoRA adapters
to predict Fantasy Premier League points per gameweek. All training was performed locally on Apple Silicon
using the MLX framework.

In [ ]:
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from pathlib import Path

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11

ROOT = Path('..').resolve()
print(f'Project root: {ROOT}')

---
## 1. The Approach: LoRA Fine-Tuning

### What is LoRA?

**Low-Rank Adaptation (LoRA)** is a technique for efficiently fine-tuning large language models. Instead of
updating all 3 billion parameters, LoRA freezes the base model and injects small trainable matrices into
specific layers. This reduces the number of trainable parameters from billions to just a few million.

Think of it like putting **specialised glasses on an existing brain**. The brain (Llama 3.2 3B) already
knows language, reasoning, numbers, and even sports. The glasses (LoRA adapter) teach it to focus that
existing knowledge on a very specific task: reading FPL player stats and outputting a single integer
prediction.

### Why Local on Apple Silicon?

Running fine-tuning locally on an M-series Mac via **MLX** has several advantages:

- **No cloud GPU costs** -- fine-tuning a 3B model fits comfortably in unified memory
- **Data stays local** -- no uploading player data to third-party services
- **Fast iteration** -- each training run takes ~15-30 minutes, enabling rapid experimentation
- **MLX is native** -- Apple's ML framework is optimised for the Metal GPU, achieving competitive throughput

### Base Model

We used `Llama 3.2 3B Instruct` quantised to 4-bit, stored at `models/llama-3.2-3b/`. The instruct variant
already understands the chat format (system/user/assistant), which means our training data can use the
same conversational structure the model was designed for.

---
## 2. Training Data Format

Training data is stored in **chat JSONL** format at `data/mlx/train.jsonl`. Each line is a complete
conversation with three messages: a system prompt defining the task, a user message with player stats,
and an assistant response with the target points.

In [ ]:
# Load and display example training prompts
train_path = ROOT / 'data' / 'mlx' / 'train.jsonl'
train_examples = []
with open(train_path) as f:
    for i, line in enumerate(f):
        if i < 3:
            train_examples.append(json.loads(line))

for idx, ex in enumerate(train_examples):
    print(f'=== Example {idx + 1} ===')
    for msg in ex['messages']:
        role = msg['role'].upper()
        content = msg['content']
        if role == 'SYSTEM':
            print(f'  [{role}] {content[:80]}...')
        elif role == 'USER':
            # Show first 3 lines + last line
            lines = content.split('\n')
            print(f'  [{role}]')
            for line in lines[:4]:
                print(f'    {line}')
            print(f'    ...')
            print(f'    {lines[-1]}')
        else:
            print(f'  [{role}] {content}')
    print()

### What the Model Sees

Each training example teaches the model a simple mapping:

1. **System**: "You are an FPL points predictor. Respond with only a single integer."
2. **User**: A structured text block with ~14 features (form, minutes, goals/90, ICT, fixture difficulty, etc.)
3. **Assistant**: A single integer -- the actual FPL points scored that gameweek

The model learns to associate patterns in the stats with point outcomes. For example, high form + easy
fixture + high ICT tends to correlate with higher points.

The training set contains **3,996 examples**, with 903 for validation and 243 for testing.

---
## 3. The Mode Collapse Problem

The biggest challenge in fine-tuning was **mode collapse**: the model learning to always predict
the same value regardless of input.

In [ ]:
# Load training data to show label distribution
train_labels = []
with open(train_path) as f:
    for line in f:
        msg = json.loads(line)
        train_labels.append(int(msg['messages'][-1]['content']))

label_counts = Counter(train_labels)

fig, ax = plt.subplots(figsize=(10, 5))
pts_range = range(min(label_counts.keys()), max(label_counts.keys()) + 1)
counts = [label_counts.get(p, 0) for p in pts_range]
bars = ax.bar(pts_range, counts, color='#4c72b0', edgecolor='white', linewidth=0.5)

# Highlight the dominant 1-2 point bucket
for bar, pt in zip(bars, pts_range):
    if pt in [1, 2]:
        bar.set_color('#c44e52')

ax.set_xlabel('Actual FPL Points')
ax.set_ylabel('Count in Training Set')
ax.set_title('Training Data Distribution (Heavily Skewed Toward 1-2 Points)')
ax.axvline(x=np.mean(train_labels), color='orange', linestyle='--', linewidth=2,
           label=f'Mean = {np.mean(train_labels):.1f}')
ax.legend()

pct_12 = (label_counts.get(1, 0) + label_counts.get(2, 0)) / len(train_labels) * 100
print(f'Points 1-2 make up {pct_12:.1f}% of the training data')
print(f'Total training examples: {len(train_labels)}')
plt.tight_layout()
plt.show()

### Why Does This Cause Mode Collapse?

FPL points follow a heavily right-skewed distribution. Most players score 1-2 points in any given
gameweek (a "blank"), while hauls of 10+ are rare. When you train a model on this distribution,
the loss function rewards it for always predicting the mode -- the model discovers that saying "2"
every time minimises the average error across the training set.

This is exactly what happened with **v1**: after 600 iterations, it learned to output "2" for
every single input. Technically a local minimum, but completely useless for prediction.

In [ ]:
# Load v3 fine-tuned predictions to show mode collapse is fixed
with open(ROOT / 'eval' / 'llm_fine_tuned_predictions.json') as f:
    ft_preds = json.load(f)

pred_values = [d['parsed_prediction'] for d in ft_preds]
actual_values = [d['actual'] for d in ft_preds]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: simulated v1 collapse (always predicts 2)
v1_preds = [2] * len(pred_values)
v1_counts = Counter(v1_preds)
axes[0].bar(v1_counts.keys(), v1_counts.values(), color='#c44e52', edgecolor='white', width=0.6)
axes[0].set_xlabel('Predicted Points')
axes[0].set_ylabel('Count')
axes[0].set_title('v1: Mode Collapse (Always Predicts 2)')
axes[0].set_xlim(-1, 12)
axes[0].set_ylim(0, 260)

# Right: v3 predictions showing range
v3_counts = Counter(pred_values)
axes[1].bar(v3_counts.keys(), v3_counts.values(), color='#55a868', edgecolor='white', width=0.6)
axes[1].set_xlabel('Predicted Points')
axes[1].set_ylabel('Count')
axes[1].set_title('v3: Fixed -- Predictions Across {' + ', '.join(str(x) for x in sorted(v3_counts.keys())) + '}')
axes[1].set_xlim(-1, 12)
axes[1].set_ylim(0, 260)

plt.suptitle('Mode Collapse: Before and After', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f'v1 unique predictions: {{2}}')
print(f'v3 unique predictions: {set(sorted(pred_values))}')

---
## 4. Three Iterations

It took three attempts to get a working fine-tuned model. Each iteration addressed specific
failures from the previous one.

In [ ]:
# Comparison table of the three adapter versions
iterations = pd.DataFrame({
    'Version': ['v1', 'v2', 'v3'],
    'Iterations': [600, 400, 500],
    'LoRA Layers': [8, 8, 16],
    'Learning Rate': ['1e-5', '2e-5', '3e-5'],
    'Rank': [8, 8, 8],
    'Data Strategy': [
        'Raw (skewed)',
        'Bucket-balanced',
        'Per-point balanced (333 each)'
    ],
    'Unique Predictions': [
        '{2}',
        '{0, 1, 2, 4}',
        '{0, 1, 2, 4, 9, 10}'
    ],
    'Outcome': [
        'Mode collapse -- always predicts 2',
        'Partial fix -- 4 values, missing high end',
        'Full range -- 6 discrete values'
    ]
})

# Style the table
styled = iterations.style.set_properties(**{
    'text-align': 'left',
    'padding': '8px'
}).set_table_styles([
    {'selector': 'th', 'props': [('text-align', 'left'), ('font-weight', 'bold')]}
]).hide(axis='index')

styled

### What Changed and Why

**v1 to v2: Data balancing.** The root cause of mode collapse was the skewed training distribution.
v2 introduced bucket-balanced sampling, grouping point values into ranges (0-1, 2-3, 4-6, 7+) and
sampling equal numbers from each bucket. This helped the model learn that outputs other than "2"
exist, but it still missed the high end of the range.

**v2 to v3: Aggressive balancing + more capacity.** v3 used per-point balanced sampling (333 examples
per point value), doubled the number of LoRA layers from 8 to 16, and increased the learning rate
to 3e-5. The combination of more balanced data and more trainable parameters allowed the model to
learn a broader range of outputs. The unique predictions expanded from 4 to 6 discrete values.

**Key insight**: The data mattered far more than the model architecture. Changing the balancing
strategy had a much bigger impact than changing layers or learning rate.

---
## 5. Prompting Strategies Comparison

We tested four strategies with the same base model to understand how prompting and fine-tuning
compare.

In [ ]:
# Load results summary
with open(ROOT / 'eval' / 'llm_results_summary.json') as f:
    results = json.load(f)

# Also load XGBoost results for reference
with open(ROOT / 'eval' / 'xgboost_results.json') as f:
    xgb_results = json.load(f)

strategies = ['zero_shot', 'few_shot', 'chain_of_thought', 'fine_tuned']
labels = ['Zero-Shot', 'Few-Shot', 'Chain-of-Thought', 'Fine-Tuned (v3)']
maes = [results[s]['mae'] for s in strategies]
xgb_mae = xgb_results['mae']

fig, ax = plt.subplots(figsize=(10, 6))

colors = ['#c44e52', '#55a868', '#4c72b0', '#8172b2']
bars = ax.bar(labels, maes, color=colors, edgecolor='white', linewidth=0.5, width=0.6)

# Add XGBoost reference line
ax.axhline(y=xgb_mae, color='orange', linestyle='--', linewidth=2,
           label=f'XGBoost MAE = {xgb_mae:.2f}')

# Add value labels on bars
for bar, mae in zip(bars, maes):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.15,
            f'{mae:.2f}', ha='center', va='bottom', fontweight='bold', fontsize=12)

ax.set_ylabel('Mean Absolute Error (lower is better)')
ax.set_title('LLM Prompting Strategies: MAE Comparison', fontsize=14, fontweight='bold')
ax.legend(fontsize=12)
ax.set_ylim(0, max(maes) * 1.15)

plt.tight_layout()
plt.show()

print(f'\nXGBoost MAE: {xgb_mae:.4f}')
for s, l in zip(strategies, labels):
    r = results[s]
    print(f'{l}: MAE={r["mae"]:.4f}, Within 1={r["within_1"]:.1%}, Within 3={r["within_3"]:.1%}')

### The Surprise: Few-Shot Nearly Matches XGBoost

The most striking result is that **few-shot prompting (MAE 2.16) nearly matches XGBoost (MAE 2.11)**
without any training at all. By simply showing the model 3-5 example predictions in the prompt,
Llama 3.2 3B was able to learn the task format and produce competitive predictions.

Meanwhile, zero-shot completely failed (MAE 11.54) -- the model had no calibration for what FPL
points look like and produced wildly inflated numbers.

Chain-of-thought performed similarly to few-shot (MAE 2.20) despite being much slower (845s vs 370s),
suggesting that explicit reasoning does not help much for this numerical prediction task.

Fine-tuned v3 (MAE 3.35) actually performed *worse* than few-shot. This is the core trade-off:
fine-tuning gave us a broader prediction range but at the cost of accuracy, because the model can
only output 6 discrete values.

---
## 6. Prediction Distribution

The fine-tuned model produces predictions from a small discrete set. Let us compare this to the
actual distribution.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Fine-tuned v3 predictions
pred_counts = Counter(pred_values)
pred_pts = sorted(pred_counts.keys())
axes[0].bar(pred_pts, [pred_counts[p] for p in pred_pts],
            color='#8172b2', edgecolor='white', linewidth=0.5, width=0.6)
axes[0].set_xlabel('Predicted Points')
axes[0].set_ylabel('Count')
axes[0].set_title('Fine-Tuned v3 Predictions')
axes[0].set_xticks(pred_pts)
for pt, cnt in pred_counts.items():
    axes[0].text(pt, cnt + 1, str(cnt), ha='center', va='bottom', fontsize=10)

# Right: Actual distribution
actual_counts = Counter(actual_values)
actual_pts = sorted(actual_counts.keys())
axes[1].bar(actual_pts, [actual_counts[p] for p in actual_pts],
            color='#4c72b0', edgecolor='white', linewidth=0.5, width=0.6)
axes[1].set_xlabel('Actual Points')
axes[1].set_ylabel('Count')
axes[1].set_title('Actual Points (Test Set)')
axes[1].set_xticks(actual_pts)

plt.suptitle('Fine-Tuned v3: Discrete Predictions vs Continuous Actuals',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f'Fine-tuned unique values: {sorted(pred_counts.keys())}')
print(f'Actual unique values:     {sorted(actual_counts.keys())}')
print(f'\nThe model compresses {len(actual_counts)} actual point values into just {len(pred_counts)} discrete buckets.')

---
## 7. Individual Predictions

Let us look at specific predictions to understand where the model succeeds and fails.
We merge with `features.csv` to get player names.

In [ ]:
# Extract player names from test.jsonl prompts
test_path = ROOT / 'data' / 'mlx' / 'test.jsonl'
test_players = []
with open(test_path) as f:
    for line in f:
        msg = json.loads(line)
        content = msg['messages'][1]['content']
        # Extract player name and position from "Player: Name (POS)"
        match = re.search(r'Player: (.+?) \((\w+)\)', content)
        if match:
            test_players.append({'name': match.group(1), 'position': match.group(2)})
        else:
            test_players.append({'name': 'Unknown', 'position': '?'})

# Build a combined dataframe
pred_df = pd.DataFrame(ft_preds)
pred_df['player'] = [p['name'] for p in test_players]
pred_df['position'] = [p['position'] for p in test_players]
pred_df['error'] = abs(pred_df['parsed_prediction'] - pred_df['actual'])

# Select a mix: 5 good predictions (low error) and 5 bad predictions (high error)
good = pred_df.nsmallest(5, 'error')[['player', 'position', 'actual', 'parsed_prediction', 'error']]
bad = pred_df.nlargest(5, 'error')[['player', 'position', 'actual', 'parsed_prediction', 'error']]

examples = pd.concat([good, bad]).reset_index(drop=True)
examples.columns = ['Player', 'Pos', 'Actual', 'Predicted', 'Error']

print('=== Best Predictions (Closest to Actual) ===')
print(examples.head(5).to_string(index=False))
print()
print('=== Worst Predictions (Furthest from Actual) ===')
print(examples.tail(5).to_string(index=False))

In [ ]:
# Visualise the 10 examples
fig, ax = plt.subplots(figsize=(12, 5))

x = range(len(examples))
width = 0.35

ax.bar([i - width/2 for i in x], examples['Actual'], width,
       label='Actual', color='#4c72b0', edgecolor='white')
ax.bar([i + width/2 for i in x], examples['Predicted'], width,
       label='Predicted', color='#8172b2', edgecolor='white')

ax.set_xticks(x)
ax.set_xticklabels([f"{r['Player']}\n({r['Pos']})" for _, r in examples.iterrows()],
                    rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Points')
ax.set_title('Individual Predictions: 5 Best (left) vs 5 Worst (right)',
             fontsize=13, fontweight='bold')
ax.axvline(x=4.5, color='gray', linestyle=':', linewidth=1)
ax.legend()

plt.tight_layout()
plt.show()

---
## 8. The Smoothing Pipeline

The fine-tuned model outputs only 6 discrete values: `{0, 1, 2, 4, 9, 10}`. This is too coarse
for a useful prediction -- we need decimal values that reflect the continuous nature of expected
points. The solution is a **blending formula** that combines the LLM signal with player features.

In [ ]:
print('Smoothing Formula:')
print('=' * 60)
print()
print('  blended = 0.40 * LLM_dampened')
print('          + 0.30 * form_3gw')
print('          + 0.30 * feature_estimate')
print()
print('Where:')
print('  - LLM_dampened = clip(raw_pred * 0.6 + 1.0, 0.5, 7.5)')
print('    Compresses the LLM output toward a realistic 0.5-7.5 range')
print()
print('  - form_3gw = recent 3-gameweek average points')
print('    The single most predictive feature in FPL')
print()
print('  - feature_estimate = weighted combination of:')
print('    form (50%) + form_5gw (30%) + ICT (15%)')
print('    + fixture adjustment + attacking bonus + clean sheet bonus')
print('    all scaled by minutes factor')
print()
print('  Final output clipped to [0.5, 10.0] and rounded to 1 decimal')

In [ ]:
# Demonstrate the smoothing effect
raw_values = sorted(Counter(pred_values).keys())

# Simulate smoothing on the raw predictions
raw_array = np.array(pred_values, dtype=float)
llm_dampened = np.clip(raw_array * 0.6 + 1.0, 0.5, 7.5)

# Use a simple form proxy (we don't have the exact features for test set here,
# but we can show the transformation)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Before: raw discrete predictions
before_counts = Counter(pred_values)
axes[0].bar(before_counts.keys(), before_counts.values(),
            color='#c44e52', edgecolor='white', width=0.6)
axes[0].set_xlabel('Points')
axes[0].set_ylabel('Count')
axes[0].set_title(f'Before Smoothing: {len(before_counts)} Unique Values')
axes[0].set_xlim(-1, 12)

# After: dampened values (showing just the LLM component transformation)
dampened_rounded = np.round(llm_dampened, 1)
after_counts = Counter(dampened_rounded)
axes[1].bar(after_counts.keys(), after_counts.values(),
            color='#55a868', edgecolor='white', width=0.3)
axes[1].set_xlabel('Points')
axes[1].set_ylabel('Count')
axes[1].set_title(f'After Dampening (LLM Component): Compressed Range')
axes[1].set_xlim(-1, 12)

plt.suptitle('Smoothing Pipeline: From Discrete to Continuous',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f'Raw unique values:      {sorted(before_counts.keys())}')
print(f'Dampened unique values:  {sorted(after_counts.keys())}')
print(f'\nAfter blending with form and features, the final output has')
print(f'continuous decimal values (e.g. 2.3, 4.7, 6.1) -- one unique value per player.')

### Why 40/30/30?

The blend weights were chosen empirically:

- **40% LLM**: The model's directional signal is valuable (it distinguishes low/mid/high performers)
  but its raw integers are too coarse. Dampening compresses the 0-10 range into 0.5-7.5.
- **30% Form**: Recent form is the single best predictor of next-gameweek points in FPL.
  Including it directly ensures the prediction stays grounded in reality.
- **30% Feature Estimate**: A weighted combination of form, ICT, fixture difficulty, and
  positional bonuses provides additional context the LLM might miss.

The result: every player gets a unique decimal prediction instead of being shoved into one of 6 buckets.

---
## 9. Key Lessons

### 1. Data Quality > Model Capacity

The jump from v1 (mode collapse) to v3 (6 unique values) was driven almost entirely by data
balancing, not model changes. Doubling LoRA layers from 8 to 16 helped, but the per-point
balanced sampling at 333 examples each was the critical fix. A model can only learn what
its training data teaches it.

### 2. Few-Shot Is Shockingly Competitive

Few-shot prompting achieved MAE 2.16 -- nearly matching XGBoost at 2.11 -- with zero training
and zero fine-tuning. This suggests that Llama 3.2 3B already has strong numerical reasoning
capabilities. For many practical applications, spending time on prompt engineering may yield
better results than spending time on fine-tuning.

### 3. Fine-Tuning Trades Accuracy for Range

The fine-tuned model (MAE 3.35) is less accurate than few-shot (MAE 2.16) on average, but it
produces a wider range of predictions including high-point hauls (9, 10). Few-shot tends to
cluster predictions in the safe 2-5 range. Whether this trade-off is worthwhile depends on
the use case -- for identifying differential captain picks, range matters more than average
accuracy.

### 4. Post-Processing Is Essential

The raw fine-tuned output ({0, 1, 2, 4, 9, 10}) is too discrete to be useful on its own.
The 40/30/30 smoothing pipeline transforms these coarse signals into realistic decimal
predictions, combining the LLM's learned intuition with concrete player features.

### 5. Local Fine-Tuning Is Practical

Running LoRA fine-tuning on Apple Silicon via MLX proved entirely practical for a 3B model.
Each training run completed in under 30 minutes, enabling rapid iteration through three
adapter versions in a single afternoon. The barrier to entry for LLM fine-tuning is lower
than ever.